In [1]:
%load_ext autoreload
%autoreload 2

```{toctree}
:maxdepth: 4
:caption: Contents:

# 1D case: walkthrough example


This notebook walks through a complete 1D XBeach case preparation using the `oceanicospy.models.xbeachpy` subpackage. The example is based on a **non-stationary** hydrodynamic simulation over a 1D cross-shore profile (Caribbean coast of Colombia, May 2025).

## Workflow overview

This example uses certain input files which must be placed in `<path_case>/input/`:

| File | Description |
|---|---|
| `profile.csv` | Two-column (x, z) cross-shore bathymetric profile |
| `SpecSWAN.out` | SWAN spectral output at the offshore boundary point |
| `points_output.txt` | (optional) list of cross-shore gauge positions |

ERA5 winds and UHSLC water levels are **downloaded automatically** if not already cached.

## Imports

In [2]:
from oceanicospy.models import xbeachpy

from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Case configuration

Three main parameters are required to set up a XBeach case:

| Parameter | Description |
|---|---|
| `path_case` | Root directory that will hold `input/`, `run/`, `output/`, `pros/` |
| `dict_ini_data` | Metadata dictionary consumed by `Initializer` |
| `ini_date` / `end_date` | Simulation window |

All user-facing settings live in plain dictionaries. The user-defined dictionary `dict_ini_data` is passed to the `Initializer` class which creates the folder structure and stamps the `params.txt` file with these settings. The expected keys for this dictionary are:

- ``case_description``: Free-form text description of the case (not used in XBeach)
- ``act_morf``: Morphological activation (0 = off)
- ``act_sedtrans``: Sediment transport activation (0 = off)
- ``act_wavemodel``: Spectral wave model activation (1 = surfbeat)
- ``dims``: Number of dimensions (**1 = 1D**)

In [3]:
# ── path to the case root ─────────────────────────────────────────────────────
path_case = '../../data/model_runs/1D_xbeach/'  # must end with a slash

# ── case-level flags written into params.txt ──────────────────────────────────
ini_case_data = dict(
    case_description='1D profile – Caribbean coast',
    act_morf=0,
    act_sedtrans=0,
    act_wavemodel=1,
    dims=1
)

# ── simulation period ─────────────────────────────────────────────────────────
ini_date = datetime(2008, 11, 20, 11)
end_date = datetime(2008, 11, 24, 21)

## Initialization

`Initializer` does two things:

1. **`create_folders`**: creates the project directory tree: `input/`, `pros/`, `run/`, and `output/` under `path_case`.  
   If `run/` or `output/` already exist they are **wiped and re-created** to avoid stale files from a previous attempt.

2. **`replace_ini_data`**: copies the bundled `params_base.txt` template into `run/params.txt` and substitutes the flags from `ini_case_data`.  
   Any key not supplied by the user falls back to the package defaults in `xbeachpy/utils/defaults.py`.

In [4]:
case = xbeachpy.Initializer(
    root_path=path_case,
    dict_ini_data=ini_case_data,
    ini_date=ini_date,
    end_date=end_date
)
case.create_folders()
case.replace_ini_data()

*** Initializing XBeach model ***


	*** Creating project folder structure ***


	*** Copying base XBeach configuration file into run folder ***



After this step the folder tree looks like:

```
path_case/
├── input/           ← place your static input files here
├── pros/
├── run/
│   └── params.txt   ← generated from the bundled template
└── output/
```

## Generating the grid

`GridMaker` can build 1D profiles or 2D rectangular grids. For a 1D case the `build_profile` property returns a builder that delegates coordinate geometry to `ProfileAxis` from the GIS module.

```{hint}
If you already have pre-built `.grd` files for x and y, place both inside `input/` and `GridMaker` will detect them automatically on instantiation.
```

In [5]:
grid_params_case = dict(thetamin=-20, thetamax=160, dtheta=180, alfa = 161)

```{hint}
The grid parameters for the frequency-direction space should be defined in a dictionary and passed to the `GridMaker` constructor. The expected keys are:
- `thetamax`: maximum wave direction (degrees)
- `thetamin`: minimum wave direction (degrees)
- `dtheta`: wave direction step (degrees)
- `alfa`: angle of the grid with respect to the x-axis (degrees)
```

In [6]:
case_grid = xbeachpy.preprocess.GridMaker(init=case,grid_params=grid_params_case)

`GridMaker` has two main properties for grid construction:

- `build_profile` — builds a 1D profile axis. This property returns a `ProfileAxis` builder with two constructors:
  - `from_coordinates(start, end, dx)` — derives the profile length from two planar endpoints (projected CRS, metres).
  - `from_length(length, dx)` — uses a scalar nominal length when coordinates are not available.
- `build_rectangular_grid` — builds a 2D rectangular grid from a shapefile bounding box (see the 2D walkthrough notebook).

Both constructors accept `dx` as either a constant `float` or a `dict` of `{breakpoint: spacing}` for spatially variable resolution.

In [7]:
# Build the profile axis from two planar endpoint coordinates (projected, metres)
profile_axis = case_grid.build_profile.from_length(
    length=3250.0,              # length of the profile [m]
    dx=5.0,                   # uniform spacing [m]
    auto_extend=True,
)

A quick inspection of `profile_axis` shows that it is a `ProfileAxis` object. The `distance_axis` property returns the cumulative cross-shore distances used by XBeach as the x grid:

In [8]:
profile_axis.distance_axis

,s
0,0.0
1,5.0
2,10.0
3,15.0
4,20.0
...,...
646,3230.0
647,3235.0
648,3240.0
649,3245.0


When the axis was built with `from_coordinates`, the corresponding planar (x, y) positions along the transect are also available:

In [9]:
# profile_axis.coordinates

If you want to check the metadata that was registered after grid generation, use the `metadata` property:

In [10]:
case_grid.metadata

{'xfilepath': 'x.grd', 'yfilepath': 'y.grd', 'meshes_x': 650, 'meshes_y': 0}

The `metadata` dictionary contains the relevant information about the grid. For a 1D profile `meshes_y` is always `0`:

```python
{
    'xfilepath': 'x.grd',
    'yfilepath': 'y.grd',
    'meshes_x' : <int>,   # number of cells in the cross-shore direction
    'meshes_y' : 0        # always 0 for 1D profiles
}
```

The grid is written to `run/` as `x.grd` (cumulative cross-shore distances) and `y.grd` (zeros). The grid section of `params.txt` is updated with the relevant metadata.

In [11]:
case_grid.fill_grid_section()


*** Adding/Editing grid information in params file ***



## Setting up the bathymetry

For a 1D case, `BathyMaker` expects a two-column (x, z) cross-shore profile file where `x` is the cross-shore distance from the seaward boundary and `z` is the bed elevation. The depths are interpolated onto the `x.grd` sampling points and written as a single-column `.dep` file.

```{note}
The 1D profile-from-file method is currently under development. Place the pre-processed `.dep` file directly in `input/` and copy it to `run/` manually, or use the 2D `convert_xyz2asc()` workflow if the input is a regular XYZ grid.
```

In [ ]:
# case_bathy = xbeachpy.preprocess.BathyMaker(init=case)
# case_bathy.build_z_from_bathy(xyz_path,ProfileAxis) (instancia de profile interpolator)
### profile_interpolator.


In [ ]:
# case_bathy = xbeachpy.preprocess.BathyMaker(init=case)
# case_bathy.build_z_from_existing_profile(xz_filepath,ProfileAxis) -> (instance de profile interpolator)

In [ ]:
# case_bathy.fill_bathy_section()

## Configuring wind forcing (ERA5)

`WindForcing` integrates directly with `oceanicospy.downloads.ERA5Downloader`. For a 1D case a small spatial window centred on the model location is sufficient since a single representative wind time series is extracted.

`WindForcing` has two main methods:
- **`get_winds_from_ERA5()`** — downloads hourly U10/V10 from the CDS API for the bounding box defined in `wind_dict`. If the NetCDF file already exists in `input/` the download is skipped.
- **`write_ERA5_ascii()`** — converts the NetCDF to a two-column (time [s], speed [m/s], direction [°N]) ASCII file and copies it into `run/`.

The `wind_dict` should contain the following keys:

- `lon_ll_corner_wind`: longitude of the lower-left corner of the wind download region
- `lat_ll_corner_wind`: latitude of the lower-left corner of the wind download region
- `nx_wind`: number of grid points in the x-direction (longitude)
- `ny_wind`: number of grid points in the y-direction (latitude)
- `dx_wind`: grid spacing in the x-direction (degrees)
- `dy_wind`: grid spacing in the y-direction (degrees)

For a 1D case a small window (`nx_wind=1, ny_wind=1`) around the profile location is enough.

In [24]:
wind_dict = dict(
    lon_ll_corner_wind=-82,
    lat_ll_corner_wind=12.567536, 
    nx_wind=1,
    ny_wind=1,
    dx_wind=0.025,
    dy_wind=0.025
)

In [25]:
case_winds = xbeachpy.preprocess.WindForcing(
    init=case,
    wind_info=wind_dict,
    use_link=False
)


*** Initializing winds ***



```{hint}
`use_link=False` copies the file; `use_link=True` creates a symlink (saves space on filesystems).
```

In [26]:
case_winds.get_winds_from_ERA5(utc_offset_hours=-5, format_localtime=True)

the file ../../data/model_runs/1D_xbeach/input/winds_era5.nc already exists
	 ERA5 wind data already exists, skipping download


When `format_localtime=True` the downloaded NetCDF timestamps are converted to local time based on `utc_offset_hours`. The output file is saved as `winds_era5_localtime.nc` in `input/`. If you use `format_localtime=False`, the timestamps remain in UTC and the file is saved as `winds_era5.nc`.

```{info}
XBeach does not use spatially distributed wind forcing. Instead it expects a single time series of wind speed and direction. `write_ERA5_ascii()` extracts the ERA5 data at the grid point nearest to `(lon_target, lat_target)` and writes it in a format suitable for XBeach.
```

In [27]:
case_winds.write_ERA5_ascii(
    era5_filename='winds_era5_localtime.nc',
    ascii_filename='winds.wnd',
    lon_target=278.326063-360,
    lat_target=12.567536
)
case_winds.fill_wind_section()

	 ERA5 wind data converted to ASCII format and saved as winds.wnd

*** Adding/Editing winds information in params file ***



## Setting up the water level forcing

`WaterLevelForcing` connects to the **University of Hawaii Sea Level Center (UHSLC)** research-quality gauge archive and downloads water level data for a specified station and time window.


The following steps are involved in setting up the water level forcing:

1. **Downloading** hourly tide-gauge data for the simulation window from UHSLC.
2. **Converting** the CSV to a two-column XBeach ASCII water-level file.
3. **Writing** the tide section of `params.txt` to point to the generated file.

The San Andres UHSLC station code is **737** according to the [UHSLC station list](https://uhslc.soest.hawaii.edu/stations/).

In [28]:
case_wl = xbeachpy.preprocess.WaterLevelForcing(init=case, use_link=False)
df_wl = case_wl.get_waterlevel_from_UHSLC(station_id=737)


*** Initializing water levels ***

Downloaded h737.csv to ../../data/model_runs/1D_xbeach/input/h737.csv.
	 UHSLC water level data was successfully downloaded


```{warning}
Unlike `WindForcing`, `get_waterlevel_from_UHSLC` returns a `pd.DataFrame` rather than writing a file directly. This allows case-specific corrections to the raw gauge record before writing the ASCII output.

For station `737` (San Andres), a known datum shift of −2 m occurred between 1997 and 2018. The mask below corrects for it:
```

In [29]:
# Correct the known −2 m datum shift for station 737 (1997–2018)
correction_mask = (
    (df_wl.index >= datetime(1997, 1, 1, 0)) &
    (df_wl.index <= datetime(2018, 12, 31, 18))
)
df_wl.loc[correction_mask, 'depth[m]'] -= 2.0

case_wl.write_UHSLC_ascii(df_wl, 'water_levels.wl')
case_wl.fill_wl_section()

	 UHSLC water level data converted to ASCII format and saved as water_levels.wl

*** Adding/Editing water level information in params file ***



## Defining the wave boundary conditions

This case uses non-stationary wave boundary conditions derived from a SWAN spectral output file (`SpecSWAN.out`). The `BoundaryConditions` class provides the `spectra_from_swan()` method that processes this file and prepares the XBeach input.

For a 1D XBeach model there is typically a **single offshore boundary point**. In this case:
- Individual `.sp2` files are written for each time step inside `run/bounds_conds/point_0/`.
- A `filelist_0.txt` references each `.sp2` file.
- A `loclist.txt` is **not** generated (it is only used for multi-point 2D boundaries). The `bcfilepath` in `params.txt` must point directly to `bounds_conds/filelist_0.txt`.

```{note}
`fill_boundaries_section()` currently only writes the boundary parameters to `params.txt` when more than one boundary point is detected (2D use case). For single-point 1D cases, update `bcfilepath` in `params.txt` manually or pass `point_indexes` with multiple points and select the desired one.
```

In [33]:
case_bounds = xbeachpy.preprocess.BoundaryConditions(init=case)

*** Cleaning Boundary Conditions ***
*** Initializing Boundary Conditions ***


Pass the index of the offshore boundary point within the SWAN output file to `point_indexes`. The index corresponds to the position of the site in the SWAN output, starting from 0.

In [35]:
case_bounds.spectra_from_swan(input_filename='SpecSWAN.out', 
                              location_points=[(0,0)],
                              point_indexes=[18])

In [32]:
case_bounds.fill_boundaries_section()


*** Adding/Editing boundary information for domain in configuration file ***



## Output and compilation configuration

`CaseRunner` finalises `params.txt` with the required output and computation settings:

| Method | Purpose |
|---|---|
| `write_output_file()` | set the output NetCDF filename |
| `write_output_points()` | read gauge positions from a text file and embed them |
| `select_global_vars()` | choose field variables written at `tintg` |
| `select_point_vars()` | choose time-series variables at gauge locations |
| `fill_computation_section()` | compute `tstop` from the start/end dates and write remaining params |

Before the compilation step define the output timing dictionary:

- `tint_value` : Output interval for time-series data at points (seconds)
- `tintg_value` : Output interval for global 1D field output (seconds)

In [36]:
# ── computation / output settings ─────────────────────────────────────────────
comp_data_nonstat = dict(
    tint_value=3600,    # output interval [s]
    tintg_value=3600,   # global output interval [s]
)

In [37]:
case_output = xbeachpy.execution.CaseRunner(
    init=case,
    dict_comp_data=comp_data_nonstat
)


*** Initializing Case Runner ***



If you want output at specific cross-shore positions, place a plain-text file in `input/` with the following format:

```
x_coordinate,y_coordinate
200,0
500,0
...
```

The coordinates must follow the model grid reference system. For this case, the file is named `points_output.txt`.

In [39]:
# filepath for the output NetCDF file
case_output.write_output_file(filename='E1_profile1D_Nov2008.nc')

# selection of variables written to the global 1D output file
case_output.select_global_vars(list_vars=['zs', 'hh', 'zb0', 'H', 'u', 'urms'])

# (optional) output points along the profile
case_output.write_output_points(filename='points_output.txt')

# selection of variables at the output points
case_output.select_point_vars(list_vars=['zs', 'H', 'urms', 'Sxx'])

In [40]:
case_output.fill_computation_section()

## Post-processing (after the run)

Once XBeach has finished, `OutputReader` lazily opens the NetCDF output and splits it into:
- **field output** — variables with more than 2 dimensions (time × x for 1D)
- **point output** — variables with exactly 2 dimensions (time × n_points)

In [ ]:
# output_path = f"{case.dict_folders['output']}profile1D.nc"

# reader = xbeachpy.postprocess.OutputReader(filepath=output_path)

# # 1D field variables  (time x x)
# field_ds = reader.read_field_output()

# # Point time-series variables  (time x n_points)
# point_ds = reader.read_point_output()

In [ ]:
# import matplotlib.pyplot as plt

# # Significant wave height across the profile at the last output time step
# H_final = field_ds['H'].isel(globaltime=-1)

# fig, ax = plt.subplots(figsize=(10, 4))
# H_final.plot(ax=ax, color='steelblue')
# ax.set_title('Significant wave height — final time step')
# ax.set_xlabel('Cross-shore distance [m]')
# ax.set_ylabel('H [m]')
# plt.tight_layout()
# plt.show()

In [ ]:
# # Water surface elevation time series at gauge 0
# zs_p0 = point_ds['zs'].isel(points=0)

# fig, ax = plt.subplots(figsize=(11, 3))
# zs_p0.plot(ax=ax, color='steelblue')
# ax.set_title('Water surface elevation — gauge 0')
# ax.set_xlabel('time')
# ax.set_ylabel('zs [m]')
# plt.tight_layout()
# plt.show()


---

### Known limitations

| # | Limitation | Location | Suggested fix |
|---|---|---|---|
| 1 | **1D bathymetry from file not yet implemented** — `BathyMaker` does not expose a `build_1d_profile_from_file()` method in the current release. Users must pre-process and copy the `.dep` file manually. | `BathyMaker` | Implement `build_1d_profile_from_file(profile_filename, x_grd_name, dep_name)` that reads a two-column CSV, interpolates onto the x grid, and writes the `.dep` file. |
| 2 | **Single-point boundary condition not handled by `fill_boundaries_section()`** — when `spectra_from_swan()` processes a single site, `dict_boundaries` is not populated and calling `fill_boundaries_section()` raises an `AttributeError`. | `BoundaryConditions.spectra_from_swan` | Add a single-point branch that sets `dict_boundaries = {'bcfilepath': 'bounds_conds/filelist_0.txt'}` and removes `n_spectrum_loc` / `loclist` entries from `params.txt`. |
| 3 | **Wind direction formula flagged as unverified** in the source code (`# this needs to be verified somehow`). The nautical convention `(270 − θ_cart) % 360` may need adjustment depending on the ERA5 u/v axis orientation at a given site. | `WindForcing._ERA5_nc_to_ascii` | Validate against a known NOAA/ECMWF record and add a unit test. |
| 4 | **`OutputReader` is minimal** — variables are separated only by array rank (ndim > 2 vs. ndim == 2). No time-slicing, spatial-subsetting, or variable-lookup helpers exist. | `postprocess.OutputReader` | Add convenience methods such as `get_var(name)`, `sel_time(t0, t1)`, and thin plotting wrappers. |
| 5 | **No CRS management** — the grid and bathymetry are assumed to share the same projected coordinate reference system. A geographic/projected mismatch produces silently incorrect output. | `GridMaker`, `BathyMaker` | Accept a `crs` argument and use `pyproj` to reproject inputs before grid assembly. |